<a href="https://colab.research.google.com/github/sreekruthy/findebate/blob/Person4/P4_Analyst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install chromadb==0.6.3 sentence-transformers nltk google-genai -q
!pip install opentelemetry-api==1.41.1 opentelemetry-sdk==1.41.1 -q

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.2/240.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependenc

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

chroma_client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/findebate_chromadb"
)
collection = chroma_client.get_collection("findebate_rag")
model = SentenceTransformer("FinLang/finance-embeddings-investopedia")

print(f"Connected. Total chunks: {collection.count()}")
# Must print: Connected. Total chunks: 6963

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Connected. Total chunks: 6963


In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

DIMENSION_QUERIES = {
    "general_financial": [
        "financial performance revenue earnings beat miss surprise results",
        "guidance outlook forecast expectations future performance strategy",
        "growth trends margin expansion profitability cash flow competitive"
    ],
    "specialized_metrics": [
        "net interest margin NIM loan deposits credit quality asset quality",
        "non-performing assets NPAs charge-offs provision loan losses",
        "return on assets ROA return on equity ROE efficiency ratio capital adequacy"
    ],
    "market_sentiment_risk": [
        "management confidence sentiment optimistic cautious positive negative tone",
        "risks challenges concerns headwinds uncertainties market conditions",
        "credit risk operational risk market risk liquidity risk hedging"
    ],
    "multi_query_integration": [
        "short-term immediate near-term weekly monthly quarterly timeline",
        "comparative performance benchmarking cross-functional analysis",
        "comprehensive integrated multi-dimensional longitudinal tracking"
    ]
}

AGENT_DIMENSION_MAP = {
    "earnings_agent"  : ["general_financial", "specialized_metrics"],
    "market_agent"    : ["general_financial", "multi_query_integration"],
    "sentiment_agent" : ["market_sentiment_risk"],
    "valuation_agent" : ["specialized_metrics", "general_financial"],
    "risk_agent"      : ["market_sentiment_risk", "specialized_metrics"]
}

def retrieve(query, top_k=5, doc_type_filter=None):
    q_emb = model.encode([query], convert_to_numpy=True).tolist()
    where_clause = {"type": doc_type_filter} if doc_type_filter else None
    results = collection.query(
        query_embeddings=q_emb,
        n_results=top_k,
        where=where_clause,
        include=["documents", "metadatas", "distances"]
    )
    output = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        output.append({
            "chunk"       : doc,
            "source_file" : meta["source_file"],
            "chunk_id"    : meta["chunk_index"],
            "type"        : meta["type"],
            "score"       : round(1 - dist, 4)
        })
    return output

def retrieve_by_dimension(dimension, doc_type_filter=None, top_k=5):
    if dimension not in DIMENSION_QUERIES:
        raise ValueError(f"Unknown dimension: {dimension}")
    seen, merged = set(), []
    for query in DIMENSION_QUERIES[dimension]:
        for r in retrieve(query, top_k=top_k, doc_type_filter=doc_type_filter):
            uid = f"{r['source_file']}_chunk_{r['chunk_id']}"
            if uid not in seen:
                seen.add(uid)
                merged.append(r)
    merged.sort(key=lambda x: x["score"], reverse=True)
    return merged[:top_k]

def get_agent_context(agent_name, source_file=None, top_k=5):
    dimensions = AGENT_DIMENSION_MAP[agent_name]
    seen, chunks = set(), []
    for dim in dimensions:
        for r in retrieve_by_dimension(dim, top_k=top_k*3):
            if source_file and r["source_file"] != source_file:
                continue
            uid = f"{r['source_file']}_chunk_{r['chunk_id']}"
            if uid not in seen:
                seen.add(uid)
                chunks.append(r)
    chunks.sort(key=lambda x: x["score"], reverse=True)
    return chunks[:top_k]

def chunks_to_text(chunks):
    return "\n\n".join([
        f"[Source: {r['source_file']} | Score: {r['score']}]\n{r['chunk']}"
        for r in chunks
    ])

print("RAG functions loaded!")

RAG functions loaded!


In [1]:
# ============================================================
# CELL 4 — GEMINI SETUP (SECURE VERSION)
# ============================================================

import json, os, time
from datetime import datetime
from google import genai
from google.genai import types

# ── Load API Key Securely ────────────────────────────────────
# This reads from Colab Secrets — key never appears in code
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    print("API key loaded from Colab Secrets!")
except Exception as e:
    # Fallback for local or HPC environment
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
    if GEMINI_API_KEY:
        print("API key loaded from environment variable!")
    else:
        print("No API key found! Add it to Colab Secrets or environment.")

# Verify key was loaded
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found! Add it to Colab Secrets.")

# ── Gemini Client ────────────────────────────────────────────
client = genai.Client(api_key=GEMINI_API_KEY)

# ── Rate Limit Tracker ───────────────────────────────────────
CALLS_PER_MINUTE = 8
call_timestamps  = []

def wait_for_rate_limit():
    global call_timestamps
    now = time.time()
    call_timestamps = [t for t in call_timestamps if now - t < 60]
    if len(call_timestamps) >= CALLS_PER_MINUTE:
        oldest    = call_timestamps[0]
        wait_time = 60 - (now - oldest) + 2
        print(f"  ⏳ Rate limit reached. Waiting {wait_time:.0f}s...")
        time.sleep(wait_time)
        now = time.time()
        call_timestamps = [t for t in call_timestamps if now - t < 60]
    call_timestamps.append(time.time())

# ── Call Function ────────────────────────────────────────────
def call_gemini(system_prompt, user_prompt, retries=3, max_tokens=3000):
    full_prompt = f"{system_prompt}\n\n{user_prompt}"
    for attempt in range(retries):
        wait_for_rate_limit()
        print(f"  Attempt {attempt+1}/{retries}...")
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=full_prompt,
                config=types.GenerateContentConfig(
                    temperature=0.6,
                    top_p=0.85,
                    max_output_tokens=max_tokens,
                    http_options=types.HttpOptions(timeout=120000)
                )
            )
            print(f"  ✅ Success!")
            return response.text
        except Exception as e:
            error_str = str(e).lower()
            if "timeout" in error_str or "timed out" in error_str:
                print(f"  ⏱ Timeout. Waiting 15s...")
                time.sleep(15)
            elif "429" in error_str or "quota" in error_str or "rate" in error_str:
                wait = 30 * (attempt + 1)
                print(f"  🚦 Rate limited. Waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"  ❌ Error: {e}")
                time.sleep(10)
    print("  💀 All attempts failed.")
    return None

# ── Quick Test ───────────────────────────────────────────────
test = call_gemini(
    "You are a financial analyst.",
    "What is DCF in one sentence?"
)
print(f"\nGemini Test: {test}")

API key loaded from Colab Secrets!
  Attempt 1/3...
  ✅ Success!

Gemini Test: DCF (Discounted Cash Flow) is a valuation method that estimates an asset's intrinsic value by discounting its projected future free cash flows back to their present value.


In [ ]:
import requests
import re

def recursive_chunk(text, max_tokens=400):
    chunks = []
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    for para in paragraphs:
        words = para.split()
        if len(words) <= max_tokens:
            if len(para) > 50:
                chunks.append(para)
        else:
            sentences = nltk.sent_tokenize(para)
            current = []
            current_len = 0
            for sent in sentences:
                sent_words = sent.split()
                if current_len + len(sent_words) <= max_tokens:
                    current.append(sent)
                    current_len += len(sent_words)
                else:
                    if current:
                        chunks.append(' '.join(current))
                    if len(sent_words) > max_tokens:
                        for i in range(0, len(sent_words), max_tokens):
                            chunk = ' '.join(sent_words[i:i+max_tokens])
                            if chunk:
                                chunks.append(chunk)
                    else:
                        current = [sent]
                        current_len = len(sent_words)
            if current:
                chunks.append(' '.join(current))
    return [c for c in chunks if len(c.strip()) > 50]


def get_all_txt_files(repo="sreekruthy/findebate", folder="data"):
    """Recursively finds all TXT files including in subfolders."""
    api_url = f"https://api.github.com/repos/{repo}/contents/{folder}"
    response = requests.get(api_url)
    items = response.json()

    txt_files = []
    for item in items:
        if item['type'] == 'file' and item['name'].endswith('.txt'):
            txt_files.append(item)
        elif item['type'] == 'dir':
            # Go into subfolder
            print(f"  Found subfolder: {item['name']}, scanning...")
            sub_url = f"https://api.github.com/repos/{repo}/contents/{item['path']}"
            sub_response = requests.get(sub_url)
            sub_items = sub_response.json()
            for sub_item in sub_items:
                if sub_item['type'] == 'file' and sub_item['name'].endswith('.txt'):
                    # Tag with company name from folder
                    sub_item['company_folder'] = item['name']
                    txt_files.append(sub_item)

    return txt_files


def index_github_files(repo="sreekruthy/findebate", folder="data"):
    print("Fetching file list from GitHub...")
    txt_files = get_all_txt_files(repo, folder)
    print(f"\nFound {len(txt_files)} TXT files total:")
    for f in txt_files:
        print(f"  - {f.get('company_folder','root')}/{f['name']}")

    total_added = 0

    for file_info in txt_files:
        filename = file_info['name']
        raw_url = file_info['download_url']
        company_folder = file_info.get('company_folder', '')

        # Determine company from folder name
        if 'apple' in company_folder.lower():
            company = 'AAPL'
        elif 'tesla' in company_folder.lower():
            company = 'TSLA'
        else:
            company = company_folder.upper()

        # source_file = company_folder/filename (without .txt)
        source_file = f"{company_folder}_{filename.replace('.txt', '')}"

        print(f"\nProcessing: {company_folder}/{filename}")
        print(f"  source_file key: '{source_file}'")

        # Check if already indexed
        try:
            existing = collection.get(
                where={"source_file": source_file},
                include=["metadatas"]
            )
            if len(existing['ids']) > 0:
                print(f"  ⚠️ Already indexed ({len(existing['ids'])} chunks). Skipping.")
                continue
        except:
            pass

        # Download
        text_response = requests.get(raw_url)
        text = text_response.text
        print(f"  Downloaded: {len(text)} characters")

        # Chunk
        chunks = recursive_chunk(text, max_tokens=400)
        print(f"  Chunked into: {len(chunks)} chunks")

        if not chunks:
            print(f"  ⚠️ No chunks, skipping.")
            continue

        # Embed
        print(f"  Embedding...")
        embeddings = model.encode(chunks, convert_to_numpy=True).tolist()

        # Prepare metadata
        ids = [f"{source_file}_chunk_{i}" for i in range(len(chunks))]
        metadatas = [
            {
                "source_file": source_file,
                "chunk_index": i,
                "type": "earnings",
                "company": company
            }
            for i in range(len(chunks))
        ]

        # Add to ChromaDB in batches
        batch_size = 100
        for i in range(0, len(chunks), batch_size):
            collection.add(
                ids=ids[i:i+batch_size],
                embeddings=embeddings[i:i+batch_size],
                documents=chunks[i:i+batch_size],
                metadatas=metadatas[i:i+batch_size]
            )

        total_added += len(chunks)
        print(f"  ✅ Added {len(chunks)} chunks for '{source_file}'")

    print(f"\n{'='*50}")
    print(f"Done! New chunks added: {total_added}")
    print(f"Total in ChromaDB now: {collection.count()}")

# Run it
index_github_files()

Fetching file list from GitHub...
  Found subfolder: apple, scanning...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


  Found subfolder: tesla, scanning...

Found 24 TXT files total:
  - apple/clean_clean_clean_earnings_q1.txt
  - apple/clean_context_q4.txt
  - apple/clean_news1.txt
  - apple/clean_news2.txt
  - apple/clean_news3.txt
  - apple/clean_news4.txt
  - apple/context_q4.txt
  - apple/earnings_q1.txt
  - apple/news1.txt
  - apple/news2.txt
  - apple/news3.txt
  - apple/news4.txt
  - tesla/clean_clean_clean_earnings_q1.txt
  - tesla/clean_context_q4.txt
  - tesla/clean_news1.txt
  - tesla/clean_news2.txt
  - tesla/clean_news3.txt
  - tesla/clean_news4.txt
  - tesla/context_q4.txt
  - tesla/earnings_q1.txt
  - tesla/news1.txt
  - tesla/news2.txt
  - tesla/news3.txt
  - tesla/news4.txt

Processing: apple/clean_clean_clean_earnings_q1.txt
  source_file key: 'apple_clean_clean_clean_earnings_q1'
  Downloaded: 49067 characters
  Chunked into: 22 chunks
  Embedding...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


  ✅ Added 22 chunks for 'apple_clean_clean_clean_earnings_q1'

Processing: apple/clean_context_q4.txt
  source_file key: 'apple_clean_context_q4'
  Downloaded: 1794 characters
  Chunked into: 1 chunks
  Embedding...
  ✅ Added 1 chunks for 'apple_clean_context_q4'

Processing: apple/clean_news1.txt
  source_file key: 'apple_clean_news1'
  Downloaded: 5855 characters
  Chunked into: 3 chunks
  Embedding...
  ✅ Added 3 chunks for 'apple_clean_news1'

Processing: apple/clean_news2.txt
  source_file key: 'apple_clean_news2'
  Downloaded: 49515 characters
  Chunked into: 22 chunks
  Embedding...
  ✅ Added 22 chunks for 'apple_clean_news2'

Processing: apple/clean_news3.txt
  source_file key: 'apple_clean_news3'
  Downloaded: 4654 characters
  Chunked into: 2 chunks
  Embedding...
  ✅ Added 2 chunks for 'apple_clean_news3'

Processing: apple/clean_news4.txt
  source_file key: 'apple_clean_news4'
  Downloaded: 2760 characters
  Chunked into: 2 chunks
  Embedding...
  ✅ Added 2 chunks for 'appl

In [ ]:
results = collection.query(
    query_embeddings=model.encode(
        ["apple tesla revenue earnings"],
        convert_to_numpy=True
    ).tolist(),
    n_results=200,
    include=["metadatas"]
)

all_sources = list(set([
    m["source_file"] for m in results["metadatas"][0]
]))
all_sources.sort()

apple_files = [s for s in all_sources if 'apple' in s.lower()]
tesla_files = [s for s in all_sources if 'tesla' in s.lower()]

print(f"Apple files: {apple_files}")
print(f"Tesla files: {tesla_files}")
print(f"Total chunks in DB: {collection.count()}")

Apple files: ['apple_clean_clean_clean_earnings_q1', 'apple_clean_context_q4', 'apple_clean_news2', 'apple_clean_news4', 'apple_context_q4', 'apple_earnings_q1', 'apple_news2', 'apple_news4']
Tesla files: ['tesla_clean_clean_clean_earnings_q1', 'tesla_clean_context_q4', 'tesla_clean_news1', 'tesla_clean_news2', 'tesla_clean_news3', 'tesla_context_q4', 'tesla_news1', 'tesla_news2', 'tesla_news3']
Total chunks in DB: 7265


In [ ]:
def run_valuation_agent(source_file=None, top_k=5):
    print(f"Running Valuation Analyst Agent on: {source_file or 'all'}...")

    chunks = get_agent_context("valuation_agent", source_file=source_file, top_k=top_k)
    context_text = chunks_to_text(chunks)

    system_prompt = """You are a CFA charterholder and senior equity research analyst with 18+ years
of experience building institutional-grade valuation assessments for major investment banks and
asset management firms. Your valuation analysis influences capital allocation decisions worth billions.

INSTITUTIONAL VALUATION AUTHORITY STANDARDS:
- Base ALL valuation assessments on verifiable business fundamentals from the earnings call
- Maintain realistic confidence levels between 70-80%
- Apply sector-specific DCF considerations
- Use dynamic weight allocation across valuation methods based on data reliability
- Focus on business quality factors verifiable from management's actual statements"""

    user_prompt = f"""Based on the following earnings call excerpts, produce an institutional-grade
valuation analysis.

EARNINGS CALL EXCERPTS:
{context_text}

VALUATION ANALYSIS FRAMEWORK (TARGET: 1000-1200 words):
1. BUSINESS QUALITY ASSESSMENT — revenue quality, competitive positioning
2. DCF VALUATION SIGNALS — growth assumptions, margin trajectory, capital efficiency
3. RELATIVE VALUATION — applicable multiples, premium/discount justification
4. FAIR VALUE DETERMINATION — bull/base/bear scenarios, 70-80% conviction range
5. INVESTMENT RECOMMENDATION — UNDERVALUED / FAIRLY VALUED / OVERVALUED

OUTPUT FORMAT — Return ONLY valid JSON, no markdown, no extra text:
{{
    "agent": "Valuation Analyst",
    "source_file": "{source_file or 'all'}",
    "timestamp": "{datetime.now().isoformat()}",
    "investment_stance": "UNDERVALUED or FAIRLY VALUED or OVERVALUED",
    "conviction_level": "75%",
    "key_points": [
        "Key valuation insight 1",
        "Key valuation insight 2",
        "Key valuation insight 3",
        "Key valuation insight 4",
        "Key valuation insight 5"
    ],
    "dcf_signals": {{
        "growth_outlook": "brief growth assessment",
        "margin_trajectory": "brief margin assessment",
        "capital_efficiency": "brief capital allocation assessment"
    }},
    "scenarios": {{
        "bull_case": "brief bull case",
        "base_case": "brief base case",
        "bear_case": "brief bear case"
    }},
    "score": 0.0,
    "reasoning": "Concise paragraph justifying stance and score"
}}

Score: 1-3=overvalued, 4-5=fairly valued, 6-8=undervalued, 9-10=strong buy"""

    raw_response = call_gemini(system_prompt, user_prompt, max_tokens=3000)
    if not raw_response:
        print("Valuation Agent failed")
        return None

    try:
        clean = raw_response.strip()
        if "```json" in clean:
            clean = clean.split("```json")[1].split("```")[0].strip()
        elif "```" in clean:
            clean = clean.split("```")[1].split("```")[0].strip()
        result = json.loads(clean)
        print(f"✅ Valuation complete! Stance: {result['investment_stance']} | Score: {result['score']}")
        return result
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        return {"agent": "Valuation Analyst", "raw": raw_response}

print("Valuation Analyst Agent loaded!")

Valuation Analyst Agent loaded!


In [ ]:
def run_risk_agent(source_file=None, top_k=5):
    print(f"Running Risk Analyst Agent on: {source_file or 'all'}...")

    chunks = get_agent_context("risk_agent", source_file=source_file, top_k=top_k)
    context_text = chunks_to_text(chunks)

    system_prompt = """You are a senior risk management specialist and former institutional portfolio
manager with extensive experience in equity risk assessment for major asset management firms.

INSTITUTIONAL RISK MANAGEMENT STANDARDS:
- Provide BALANCED risk assessment — avoid excessive pessimism and unwarranted optimism
- Evaluate across THREE core dimensions: credit risk, interest rate risk, liquidity risk
- Support ALL risk evaluations with specific content from the actual earnings call
- Maintain realistic confidence levels between 70-80%
- Deliver actionable position sizing guidance based on risk profile"""

    user_prompt = f"""Based on the following earnings call excerpts, produce an institutional-grade
risk assessment.

EARNINGS CALL EXCERPTS:
{context_text}

RISK ASSESSMENT FRAMEWORK (TARGET: 800-1000 words):
1. CREDIT RISK — asset quality, provisioning, stress scenarios
2. INTEREST RATE RISK — rate sensitivity, NIM impact, hedging strategies
3. LIQUIDITY RISK — funding stability, deposit quality, liquidity buffers
4. OPERATIONAL & MARKET RISK — competitive headwinds, regulatory risks
5. RISK-ADJUSTED POSITION SIZING — overall rating, recommended position size

OUTPUT FORMAT — Return ONLY valid JSON, no markdown, no extra text:
{{
    "agent": "Risk Analyst",
    "source_file": "{source_file or 'all'}",
    "timestamp": "{datetime.now().isoformat()}",
    "overall_risk_rating": "LOW or MODERATE or HIGH or VERY HIGH",
    "conviction_level": "75%",
    "key_points": [
        "Key risk insight 1",
        "Key risk insight 2",
        "Key risk insight 3",
        "Key risk insight 4",
        "Key risk insight 5"
    ],
    "risk_dimensions": {{
        "credit_risk": {{
            "rating": "LOW or MODERATE or HIGH",
            "assessment": "brief assessment"
        }},
        "interest_rate_risk": {{
            "rating": "LOW or MODERATE or HIGH",
            "assessment": "brief assessment"
        }},
        "liquidity_risk": {{
            "rating": "LOW or MODERATE or HIGH",
            "assessment": "brief assessment"
        }},
        "operational_risk": {{
            "rating": "LOW or MODERATE or HIGH",
            "assessment": "brief assessment"
        }}
    }},
    "position_sizing": {{
        "recommended_position": "X% of portfolio",
        "max_position": "X% of portfolio",
        "hedge_strategies": ["strategy 1", "strategy 2"]
    }},
    "risk_triggers": [
        "Trigger 1",
        "Trigger 2",
        "Trigger 3"
    ],
    "score": 0.0,
    "reasoning": "Concise paragraph justifying overall risk rating and score"
}}

Score: 1-3=very high risk, 4-5=moderate risk, 6-8=manageable risk, 9-10=low risk"""

    raw_response = call_gemini(system_prompt, user_prompt, max_tokens=3000)
    if not raw_response:
        print("Risk Agent failed")
        return None

    try:
        clean = raw_response.strip()
        if "```json" in clean:
            clean = clean.split("```json")[1].split("```")[0].strip()
        elif "```" in clean:
            clean = clean.split("```")[1].split("```")[0].strip()
        result = json.loads(clean)
        print(f"✅ Risk complete! Rating: {result['overall_risk_rating']} | Score: {result['score']}")
        return result
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        return {"agent": "Risk Analyst", "raw": raw_response}

print("Risk Analyst Agent loaded!")

Risk Analyst Agent loaded!


In [ ]:
def run_report_synthesizer(agent_outputs, source_file=None):
    print(f"Running Report Synthesizer on: {source_file or 'all'}...")

    agents_summary = json.dumps(agent_outputs, indent=2)

    system_prompt = """You are a Managing Director at a top-tier investment bank crafting
institutional investment reports. Portfolio managers will make Long/Short decisions for
1-day, 1-week, and 1-month timeframes based on your analysis.

PROFESSIONAL STANDARDS:
- Synthesize ALL analyst perspectives into one coherent narrative
- Maintain realistic conviction levels between 70-80%
- Ground every claim in verifiable earnings call evidence
- Never contradict yourself across timeframes
- Produce institutional-grade language throughout"""

    user_prompt = f"""Synthesize these analyst outputs into a single institutional investment report.

ANALYST OUTPUTS:
{agents_summary}

OUTPUT FORMAT — Return ONLY valid JSON, no markdown, no extra text:
{{
    "agent": "Report Synthesizer",
    "source_file": "{source_file or 'all'}",
    "timestamp": "{datetime.now().isoformat()}",
    "overall_stance": "BULLISH or NEUTRAL or BEARISH",
    "overall_conviction": "75%",
    "executive_summary": "150 word summary here",
    "multi_dimensional_synthesis": {{
        "earnings_highlights": "key earnings insights",
        "market_positioning": "market dynamics summary",
        "management_sentiment": "sentiment assessment",
        "valuation_summary": "valuation conclusion",
        "risk_profile": "overall risk summary"
    }},
    "investment_recommendations": {{
        "one_day": {{
            "position": "LONG or SHORT or NEUTRAL",
            "conviction": "75%",
            "expected_direction": "brief direction",
            "key_catalyst": "specific catalyst"
        }},
        "one_week": {{
            "position": "LONG or SHORT or NEUTRAL",
            "conviction": "75%",
            "expected_direction": "brief direction",
            "momentum_drivers": "key drivers"
        }},
        "one_month": {{
            "position": "LONG or SHORT or NEUTRAL",
            "conviction": "75%",
            "expected_direction": "brief direction",
            "fundamental_catalysts": "key catalysts"
        }}
    }},
    "risk_reward": {{
        "upside_catalysts": ["catalyst 1", "catalyst 2", "catalyst 3"],
        "downside_risks": ["risk 1", "risk 2", "risk 3"],
        "position_sizing": "X% of portfolio",
        "hedge_strategies": ["hedge 1", "hedge 2"]
    }},
    "investment_conclusion": {{
        "final_stance": "BULLISH or NEUTRAL or BEARISH",
        "conviction": "75%",
        "top_3_insights": [
            "Actionable insight 1",
            "Actionable insight 2",
            "Actionable insight 3"
        ]
    }},
    "agent_scores_summary": {{
        "valuation_score": 0.0,
        "risk_score": 0.0,
        "composite_score": 0.0
    }},
    "reasoning": "Concise paragraph explaining synthesis"
}}"""

    raw_response = call_gemini(system_prompt, user_prompt, max_tokens=4000)
    if not raw_response:
        print("Synthesizer failed")
        return None

    try:
        clean = raw_response.strip()
        if "```json" in clean:
            clean = clean.split("```json")[1].split("```")[0].strip()
        elif "```" in clean:
            clean = clean.split("```")[1].split("```")[0].strip()
        result = json.loads(clean)
        print(f"✅ Synthesis complete!")
        print(f"   Stance : {result['overall_stance']}")
        print(f"   1-Day  : {result['investment_recommendations']['one_day']['position']}")
        print(f"   1-Week : {result['investment_recommendations']['one_week']['position']}")
        print(f"   1-Month: {result['investment_recommendations']['one_month']['position']}")
        return result
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        return {"agent": "Report Synthesizer", "raw": raw_response}

print("Report Synthesizer Agent loaded!")

Report Synthesizer Agent loaded!


In [ ]:
def save_results(source_file, valuation, risk, synthesis):
    output_dir = "/content/drive/MyDrive/findebate_outputs"
    os.makedirs(output_dir, exist_ok=True)

    output = {
        "source_file": source_file,
        "timestamp": datetime.now().isoformat(),
        "agents": {
            "valuation": valuation,
            "risk": risk,
            "synthesis": synthesis
        }
    }

    filename = f"{output_dir}/{source_file}_p4_output.json"
    with open(filename, "w") as f:
        json.dump(output, f, indent=2)

    print(f"✅ Saved to: {filename}")
    return filename

print("Save function loaded!")

Save function loaded!


In [ ]:
def get_all_source_files():
    results = collection.query(
        query_embeddings=model.encode(
            ["financial earnings revenue"],
            convert_to_numpy=True
        ).tolist(),
        n_results=500,
        include=["metadatas"]
    )
    source_files = list(set([
        meta["source_file"]
        for meta in results["metadatas"][0]
    ]))
    source_files.sort()
    return source_files

all_files = get_all_source_files()
print(f"Total unique transcripts: {len(all_files)}")
for i, f in enumerate(all_files):
    print(f"  {i+1}. {f}")

Total unique transcripts: 68
  1. ABM_q3_2021
  2. AME_q1_2021
  3. CFR_q3_2019
  4. CMI_q1_2014
  5. CMI_q1_2015
  6. CMI_q3_2014
  7. CMI_q4_2013
  8. CMI_q4_2014
  9. CMI_q4_2015
  10. CPF_q4_2019
  11. DE_q1_2014
  12. DE_q2_2014
  13. DE_q3_2013
  14. DE_q3_2014
  15. DE_q4_2012
  16. DE_q4_2014
  17. DNB_q2_2021
  18. DOV_q2_2020
  19. DX_q1_2021
  20. ETN_q1_2014
  21. FAF_q4_2020
  22. FIS_q4_2020
  23. FN_q2_2021
  24. FSS_q2_2021
  25. GCO_q1_2022
  26. GD_q1_2021
  27. GLW_q1_2021
  28. GNW_q2_2021
  29. HR_q2_2021
  30. HTH_q4_2020
  31. JBL_q4_2021
  32. KMT_q1_2022
  33. KW_q3_2021
  34. LH_q3_2021
  35. LNN_q3_2021
  36. LYB_q3_2021
  37. MDT_q2_2022
  38. MKC_q2_2021
  39. MSI_q3_2021
  40. MYE_q2_2020
  41. NEE_q3_2021
  42. NPO_q2_2021
  43. OHI_q4_2021
  44. PCAR_q1_2014
  45. PCAR_q1_2015
  46. PCAR_q1_2016
  47. PCAR_q2_2014
  48. PCAR_q2_2015
  49. PCAR_q3_2014
  50. PCAR_q3_2015
  51. PCAR_q4_2014
  52. PCAR_q4_2015
  53. RPM_q3_2022
  54. SF_q3_2020
  55. SWN_q2

In [ ]:
def run_full_pipeline(source_file, top_k=5):
    print(f"\n{'='*60}")
    print(f"Processing: {source_file}")
    print(f"{'='*60}")

    try:
        # Valuation Agent
        print("\n[1/3] Valuation Agent...")
        valuation = run_valuation_agent(source_file=source_file, top_k=top_k)

        # Risk Agent
        print("\n[2/3] Risk Agent...")
        risk = run_risk_agent(source_file=source_file, top_k=top_k)

        # Synthesizer
        print("\n[3/3] Synthesizer...")
        valid_outputs = [o for o in [valuation, risk] if o is not None]
        synthesis = run_report_synthesizer(
            agent_outputs=valid_outputs,
            source_file=source_file
        )

        # Save
        if valuation and risk and synthesis:
            save_results(source_file, valuation, risk, synthesis)
            return {
                "source_file": source_file,
                "status": "success",
                "stance": synthesis.get("overall_stance"),
                "1day": synthesis["investment_recommendations"]["one_day"]["position"],
                "1week": synthesis["investment_recommendations"]["one_week"]["position"],
                "1month": synthesis["investment_recommendations"]["one_month"]["position"],
            }
        else:
            return {"source_file": source_file, "status": "partial_failure"}

    except Exception as e:
        print(f"❌ Error: {e}")
        return {"source_file": source_file, "status": "failed", "error": str(e)}

print("Pipeline runner loaded!")

Pipeline runner loaded!


In [ ]:
# Test on Apple earnings call first
TEST_FILE = "apple_earnings_q1"

print(f"Testing full pipeline on: {TEST_FILE}")
result = run_full_pipeline(TEST_FILE, top_k=5)
print(f"\nFinal Result:")
print(json.dumps(result, indent=2))

Testing full pipeline on: apple_earnings_q1

Processing: apple_earnings_q1

[1/3] Valuation Agent...
Running Valuation Analyst Agent on: apple_earnings_q1...
  Attempt 1/3...
  ✅ Success!
JSON parse error: Unterminated string starting at: line 9 column 9 (char 325)

[2/3] Risk Agent...
Running Risk Analyst Agent on: apple_earnings_q1...
  Attempt 1/3...
  ✅ Success!
JSON parse error: Unterminated string starting at: line 37 column 13 (char 6953)

[3/3] Synthesizer...
Running Report Synthesizer on: apple_earnings_q1...
  Attempt 1/3...
  ✅ Success!
✅ Synthesis complete!
   Stance : NEUTRAL
   1-Day  : NEUTRAL
   1-Week : NEUTRAL
   1-Month: NEUTRAL
✅ Saved to: /content/drive/MyDrive/findebate_outputs/apple_earnings_q1_p4_output.json

Final Result:
{
  "source_file": "apple_earnings_q1",
  "status": "success",
  "stance": "NEUTRAL",
  "1day": "NEUTRAL",
  "1week": "NEUTRAL",
  "1month": "NEUTRAL"
}


In [ ]:
# Test Tesla
result_tesla = run_full_pipeline("tesla_clean_clean_clean_earnings_q1", top_k=5)
print(json.dumps(result_tesla, indent=2))


Processing: tesla_clean_clean_clean_earnings_q1

[1/3] Valuation Agent...
Running Valuation Analyst Agent on: tesla_clean_clean_clean_earnings_q1...
  Attempt 1/3...
  ✅ Success!
✅ Valuation complete! Stance: N/A - INPUT MISSING | Score: 0.0

[2/3] Risk Agent...
Running Risk Analyst Agent on: tesla_clean_clean_clean_earnings_q1...
  Attempt 1/3...
  ✅ Success!
JSON parse error: Unterminated string starting at: line 29 column 27 (char 3431)

[3/3] Synthesizer...
Running Report Synthesizer on: tesla_clean_clean_clean_earnings_q1...
  Attempt 1/3...
  ✅ Success!
✅ Synthesis complete!
   Stance : NEUTRAL
   1-Day  : NEUTRAL
   1-Week : NEUTRAL
   1-Month: NEUTRAL
✅ Saved to: /content/drive/MyDrive/findebate_outputs/tesla_clean_clean_clean_earnings_q1_p4_output.json
{
  "source_file": "tesla_clean_clean_clean_earnings_q1",
  "status": "success",
  "stance": "NEUTRAL",
  "1day": "NEUTRAL",
  "1week": "NEUTRAL",
  "1month": "NEUTRAL"
}


In [ ]:
# should run on hpc
def run_batch(file_list, top_k=5, start_from=0):
    batch_results = []
    failed_files = []
    total = len(file_list)
    files_to_run = file_list[start_from:]

    print(f"Starting batch: {len(files_to_run)} transcripts")

    for i, source_file in enumerate(files_to_run):
        actual_index = start_from + i
        print(f"\nProgress: {actual_index+1}/{total}")

        result = run_full_pipeline(source_file, top_k=top_k)
        batch_results.append(result)

        if result["status"] != "success":
            failed_files.append(source_file)

        # Checkpoint every 5 files
        if (i + 1) % 5 == 0:
            checkpoint = {
                "timestamp": datetime.now().isoformat(),
                "completed": actual_index + 1,
                "total": total,
                "results": batch_results,
                "failed": failed_files
            }
            with open("/content/drive/MyDrive/findebate_outputs/checkpoint.json", "w") as f:
                json.dump(checkpoint, f, indent=2)
            print(f"💾 Checkpoint saved at {actual_index+1}/{total}")

    # Final summary
    summary = {
        "timestamp": datetime.now().isoformat(),
        "total": total,
        "successful": len([r for r in batch_results if r["status"] == "success"]),
        "failed": len(failed_files),
        "failed_files": failed_files,
        "results": batch_results
    }
    with open("/content/drive/MyDrive/findebate_outputs/batch_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print(f"\n{'='*60}")
    print(f"BATCH COMPLETE")
    print(f"Successful: {summary['successful']}/{total}")
    print(f"Failed    : {summary['failed']}/{total}")
    return summary

# ── Run sample (today) ────────────────────────────────────────
# Uncomment ONE of these:

# Sample run — 2 files only
# sample = run_batch(all_files[:2], top_k=3)

# Full batch — ALL files (run on HPC Monday)
# full = run_batch(all_files, top_k=5)

# Resume from checkpoint if interrupted
# resumed = run_batch(all_files, top_k=5, start_from=10)

print("Batch runner loaded! Uncomment a line above to run.")

Batch runner loaded! Uncomment a line above to run.


In [ ]:
# Check your output files saved correctly
import os
output_dir = "/content/drive/MyDrive/findebate_outputs"
files = os.listdir(output_dir)
print("Saved outputs:")
for f in files:
    print(f"  - {f}")

Saved outputs:
  - CMI_q1_2015_p4_output.json
  - apple_earnings_q1_p4_output.json
  - tesla_clean_clean_clean_earnings_q1_p4_output.json
